In [0]:
from pyspark.sql import functions as F

batch_id = "batch_001"
pdf_folder = (
    f"/Volumes/workspace/migration_dev/landing/{batch_id}/pdfs"
)
target_table = "workspace.migration_dev.bronze_file_inventory"

files = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(pdf_folder)
)

inventory = files.select(
    F.lit(batch_id).alias("batch_id"),
    F.regexp_extract("path", r"([^/]+)$", 1).alias("file_name"),
    F.regexp_replace("path", r"^dbfs:", "").alias("pdf_path"),
    F.col("length").alias("file_size_bytes"),
    F.col("modificationTime").alias("file_modification_time"),
    F.sha2("content", 256).alias("actual_sha256"),
    F.current_timestamp().alias("inventory_timestamp"),
)

# Refresh only this batch's inventory on each run.
# Other batches in the table remain unchanged.
(
    inventory.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"batch_id = '{batch_id}'")
    .saveAsTable(target_table)
)

saved_inventory = (
    spark.table(target_table)
    .filter(F.col("batch_id") == batch_id)
)

print(f"PDFs inventoried: {saved_inventory.count()}")
display(saved_inventory.orderBy("file_name").limit(5))

PDFs inventoried: 100


batch_id,file_name,pdf_path,file_size_bytes,file_modification_time,actual_sha256,inventory_timestamp
batch_001,DOC000001.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000001.pdf,1542,2026-10-13T14:39:05.000Z,12298c6adcbc1f0ffed046f3c14ab064da327b4fab519c9e25014e6a8182c431,2026-10-14T07:17:14.831Z
batch_001,DOC000002.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000002.pdf,1542,2026-10-13T14:39:05.000Z,e45b766bfe7a3f38c7303239ffab722e244474ccb300ef84529b48197cbb83ce,2026-10-14T07:17:14.831Z
batch_001,DOC000003.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000003.pdf,1543,2026-10-13T14:39:05.000Z,eecd9ffaf648015b1dcc0a49067b9724e9a38b609a1174a19046bf6d33a662c7,2026-10-14T07:17:14.831Z
batch_001,DOC000004.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000004.pdf,1542,2026-10-13T14:39:05.000Z,8a1b437face5c2d7feb3fbee111a1b98b3f72afe81110e92fa55279fc7e3ea76,2026-10-14T07:17:14.831Z
batch_001,DOC000005.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000005.pdf,1542,2026-10-13T14:39:05.000Z,e9907b9c7b5e0b6d0836ae782f5a5de81780eb22dbba4802102a97ed69773eb6,2026-10-14T07:17:14.831Z
